# **Motor de Evaluación y Pruebas de Cordura**

| **Integrante 3:** *[Tu nombre]*

Este notebook cubre tres responsabilidades principales:

1. **Prueba de Cordura (Sanity Check):** Verificación numérica por diferencias finitas de que los gradientes analíticos derivados son matemáticamente correctos (error relativo < 10⁻⁴).
2. **Algoritmo 2 — Evaluación en Datos No Vistos:** Dado el diccionario espectral `W` aprendido durante el entrenamiento, resolver el problema de mínimos cuadrados no negativos para obtener `H_eval` en los conjuntos de validación y prueba.
3. **Métrica RMSE:** Calcular el error de reconstrucción espectral para comparar el desempeño de los distintos optimizadores y valores de `k`.

---
## **Bloque 0: Importaciones y Configuración**

In [1]:
import numpy as np
import sys
import os

# Agregar el directorio src al path para poder importar el algoritmo del Integrante 2
# Ajusta esta ruta si tu estructura de carpetas es diferente
sys.path.append(os.path.join("..", "BCGD"))

from algoritmoBCGDUnificado import bcgd_matrix_factorization


---
## **Bloque 1: Prueba de Cordura — Verificación Numérica del Gradiente**

### **Justificación Teórica**

Antes de confiar en cualquier algoritmo de optimización basado en gradientes, debemos asegurarnos de que la derivada analítica que calculamos a mano es correcta. Para esto, usamos la **aproximación de diferencias finitas hacia adelante**:

$$\frac{\partial f}{\partial W_{ab}} \approx \frac{f(W + \varepsilon E_{ab}) - f(W)}{\varepsilon}$$

donde $E_{ab}$ es una matriz con un `1` en la posición $(a,b)$ y ceros en el resto, y $\varepsilon$ es una perturbación pequeña (típicamente $10^{-5}$).

La función de pérdida del modelo NMF es:
$$f(W, H) = \frac{1}{2} \|X - WH\|_F^2$$

cuyo gradiente analítico respecto a $W$ es:
$$\nabla_W f = (WH - X)H^T$$

El criterio de éxito es que el **error relativo** entre ambos gradientes sea inferior a $10^{-4}$:

$$\text{Error Relativo} = \frac{\|\nabla_W^{\text{analítico}} - \nabla_W^{\text{numérico}}\|_2}{\|\nabla_W^{\text{analítico}}\|_2} < 10^{-4}$$

In [2]:
# ============================================================
# PRUEBA DE CORDURA: Diferencias Finitas vs. Gradiente Analítico
# Ejemplo pequeño con m=2, n=2, k=1 para facilitar la inspección.
# ============================================================

# --- Configuración del ejemplo mínimo ---
m, n, k = 2, 2, 1
X = np.array([[5., 3.],
              [2., 4.]])
W = np.array([[1.],
              [2.]])
H = np.array([[2., 1.]])

# --- 1. Gradiente Analítico ---
# Fórmula derivada: nabla_W f = (WH - X) H^T
grad_W_analytical = (W @ H - X) @ H.T

# --- 2. Gradiente Numérico (Diferencias Finitas Hacia Adelante) ---
epsilon = 1e-5
grad_W_numerical = np.zeros_like(W)

def loss_nmf(W_val, H_val, X_val):
    """Función de pérdida base sin máscara: (1/2) * ||X - WH||_F^2"""
    return 0.5 * np.sum((X_val - W_val @ H_val) ** 2)

f0 = loss_nmf(W, H, X)  # Pérdida base

for a in range(m):
    for b in range(k):
        E = np.zeros_like(W)
        E[a, b] = 1.0
        grad_W_numerical[a, b] = (loss_nmf(W + epsilon * E, H, X) - f0) / epsilon

# --- 3. Cálculo del Error Relativo ---
error_relativo = (np.linalg.norm(grad_W_analytical - grad_W_numerical) /
                  np.linalg.norm(grad_W_analytical))

# --- 4. Reporte de Resultados ---
print("=" * 52)
print("  VERIFICACIÓN DE GRADIENTES (Sanity Check)")
print("=" * 52)
print(f"\nGradiente Analítico  nabla_W f = (WH-X)H^T:")
print(grad_W_analytical)
print(f"\nGradiente Numérico  (diferencias finitas):")
print(grad_W_numerical)
print(f"\nError Relativo: {error_relativo:.8e}")
print("-" * 52)

if error_relativo < 1e-4:
    print("RESULTADO: ✓ ÉXITO — Error relativo < 1e-4")
    print("La derivación analítica del equipo es correcta.")
else:
    print("RESULTADO: ✗ ERROR — El error relativo es demasiado alto.")
    print("Revisar la derivación o la implementación del gradiente.")

print("=" * 52)

  VERIFICACIÓN DE GRADIENTES (Sanity Check)

Gradiente Analítico  nabla_W f = (WH-X)H^T:
[[-8.]
 [ 2.]]

Gradiente Numérico  (diferencias finitas):
[[-7.999975]
 [ 2.000025]]

Error Relativo: 4.28746923e-06
----------------------------------------------------
RESULTADO: ✓ ÉXITO — Error relativo < 1e-4
La derivación analítica del equipo es correcta.


---
## **Bloque 2: Algoritmo 2 — Evaluación en Datos No Vistos**

### **Justificación Teórica**

Una vez que el entrenamiento converge, obtenemos un diccionario espectral $W \in \mathbb{R}_{\geq 0}^{F \times k}$ que captura los patrones de frecuencia del audio. Para evaluar qué tan bien generaliza el modelo en datos no vistos (val/test), **fijamos $W$** y resolvemos el siguiente subproblema de mínimos cuadrados no negativos (NNLS) para cada conjunto:

$$H_{\text{eval}} = \arg\min_{H \geq 0} \frac{1}{2} \|X_{\text{eval}} - W H\|_F^2$$

Este es exactamente el mismo subproblema del bloque $H$ dentro del BCGD, con la diferencia de que $W$ está fijo. Lo resolvemos ejecutando el BCGD **solo sobre el bloque H** (`innerW=0`), usando como datos de entrada la matriz del conjunto correspondiente.

In [3]:
def solve_H_eval(W_fixed, X_eval, k, steps=300, alphaH=1e-3, method='gd', beta=0.9):
    """
    Algoritmo 2: Resuelve para H_eval fijando el diccionario W aprendido.
    
    Dado W fijo (aprendido en entrenamiento), encuentra la matriz de activaciones
    H_eval que minimiza (1/2)||X_eval - W @ H_eval||^2, sujeto a H_eval >= 0.
    
    Parámetros
    ----------
    W_fixed : ndarray de shape (F, k)
        Diccionario espectral aprendido durante el entrenamiento. Se mantiene fijo.
    X_eval : ndarray de shape (F, T_eval)
        Espectrograma del conjunto de evaluación (val o test).
    k : int
        Número de componentes (rango de la factorización).
    steps : int
        Número de iteraciones del optimizador sobre H.
    alphaH : float
        Tasa de aprendizaje para el bloque H.
    method : str
        Variante del optimizador: 'gd', 'momentum' o 'nesterov'.
    beta : float
        Coeficiente de momentum (solo relevante si method != 'gd').
    
    Retorna
    -------
    H_eval : ndarray de shape (k, T_eval)
        Activaciones temporales optimizadas para el conjunto de evaluación.
    loss_history : list
        Historia de la pérdida de reconstrucción a lo largo de las iteraciones.
    """
    F, T_eval = X_eval.shape
    
    # Inicialización aleatoria de H_eval (positiva, escalada por 1/sqrt(k))
    np.random.seed(0)
    H_eval = np.random.uniform(0, 1 / np.sqrt(k), (k, T_eval))
    
    # Velocidad de momentum inicializada en cero
    vH = np.zeros_like(H_eval)
    loss_history = []
    
    for s in range(steps):
        if method == 'nesterov':
            # Look-ahead de Nesterov
            H_look = H_eval - alphaH * beta * vH
            H_look = np.maximum(H_look, 0)  # Proyección intermedia
            R_look = W_fixed @ H_look - X_eval
            gH = W_fixed.T @ R_look
            vH = beta * vH + gH
            H_eval = H_eval - alphaH * vH
        else:
            # Gradiente estándar del bloque H con W fijo
            R = W_fixed @ H_eval - X_eval
            gH = W_fixed.T @ R
            if method == 'gd':
                H_eval = H_eval - alphaH * gH
            elif method == 'momentum':
                vH = beta * vH + gH
                H_eval = H_eval - alphaH * vH
        
        # Proyección de no negatividad (restricción NMF)
        H_eval = np.maximum(H_eval, 0)
        
        # Registro de la pérdida en esta iteración
        loss = 0.5 * np.sum((W_fixed @ H_eval - X_eval) ** 2)
        loss_history.append(loss)
    
    return H_eval, loss_history



---
## **Bloque 3: Métrica de Error — RMSE**

### **Justificación Teórica**

El **RMSE (Root Mean Square Error)** mide el error de reconstrucción espectral entre el espectrograma original $X$ y la reconstrucción del modelo $\hat{X} = WH$:

$$\text{RMSE} = \sqrt{\frac{1}{F \cdot T} \sum_{f,t} (X_{ft} - \hat{X}_{ft})^2} = \frac{\|X - WH\|_F}{\sqrt{F \cdot T}}$$

Un RMSE más bajo indica una mejor reconstrucción espectral. Esta métrica nos permite comparar cuantitativamente los optimizadores y seleccionar el valor óptimo de $k$.

In [4]:
def compute_rmse(X, W, H):
    """
    Calcula el RMSE de reconstrucción espectral entre X y W @ H.
    
    Parámetros
    ----------
    X : ndarray de shape (F, T)
        Espectrograma original (ground truth).
    W : ndarray de shape (F, k)
        Diccionario espectral aprendido.
    H : ndarray de shape (k, T)
        Activaciones temporales (entrenamiento o evaluación).
    
    Retorna
    -------
    rmse : float
        Raíz del error cuadrático medio de reconstrucción.
    """
    F, T = X.shape
    reconstruction_error = np.linalg.norm(X - W @ H, 'fro')
    rmse = reconstruction_error / np.sqrt(F * T)
    return rmse



---
## **Bloque 4: Pipeline de Evaluación Completo**

Este bloque integra todo el flujo: toma las matrices del Integrante 1 (`X_train`, `X_val`, `X_test`) y el modelo entrenado por el Integrante 2, y produce los RMSE de validación y prueba para cada configuración de optimizador.

In [5]:


# --- Opción A: Cargar matrices guardadas en disco ---
X_train = np.load("../data/processed/X_train.npy")
X_val   = np.load("../data/processed/X_val.npy")
X_test  = np.load("../data/processed/X_test.npy")

# --- Opción B: DESACTIVADA ---
# np.random.seed(7)
# F_dim = 1025
# X_full_synthetic = np.abs(np.random.randn(F_dim, 200))
# T = X_full_synthetic.shape[1]
# train_end = int(T * 0.70)
# val_end   = train_end + int(T * 0.15)
# X_train = X_full_synthetic[:, :train_end]
# X_val   = X_full_synthetic[:, train_end:val_end]
# X_test  = X_full_synthetic[:, val_end:]

print(f"Matrices cargadas:")
print(f"  X_train : {X_train.shape}")
print(f"  X_val   : {X_val.shape}")
print(f"  X_test  : {X_test.shape}")

Matrices cargadas:
  X_train : (1025, 886)
  X_val   : (1025, 190)
  X_test  : (1025, 191)


In [6]:
# ============================================================
# EVALUACIÓN: Entrenamiento + Evaluación por optimizador
# Se itera sobre los tres métodos del Algoritmo 1 (Integrante 2)
# y se reportan los RMSE en validación y prueba.
# ============================================================

k            = 10      # Rango de factorización
steps_train  = 500     # Iteraciones de entrenamiento
steps_eval   = 300     # Iteraciones para resolver H_eval
beta         = 0.9     # Coeficiente de momentum

# Nesterov requiere un alpha más pequeño para no diverger con datos reales
configs = {
    'gd':       {'alphaW': 1e-3, 'alphaH': 1e-3},
    'momentum': {'alphaW': 1e-3, 'alphaH': 1e-3},
    'nesterov': {'alphaW': 1e-5, 'alphaH': 1e-5},
}

methods = ['gd', 'momentum', 'nesterov']
results = {}

for method in methods:
    print(f"\n{'='*55}")
    print(f"  Optimizador: {method.upper()}")
    print(f"{'='*55}")
    
    # --- Paso 1: Entrenar el modelo con X_train (Algoritmo 1) ---
    W_learned, H_train, loss_hist = bcgd_matrix_factorization(
        X      = X_train,
        k      = k,
        steps  = steps_train,
        alphaW = configs[method]['alphaW'],
        alphaH = configs[method]['alphaH'],
        method = method,
        beta   = beta,
        lambd  = 0.0
    )
    
    rmse_train = compute_rmse(X_train, W_learned, H_train)
    print(f"  Loss final (entrenamiento): {loss_hist[-1]:.4f}")
    print(f"  RMSE train               : {rmse_train:.6f}")
    
    # --- Paso 2: Resolver H_val fijando W aprendida (Algoritmo 2) ---
    H_val, _ = solve_H_eval(
        W_fixed = W_learned,
        X_eval  = X_val,
        k       = k,
        steps   = steps_eval,
        alphaH  = configs[method]['alphaH'],
        method  = method,
        beta    = beta
    )
    
    rmse_val = compute_rmse(X_val, W_learned, H_val)
    print(f"  RMSE val                 : {rmse_val:.6f}")
    
    # --- Paso 3: Resolver H_test fijando W aprendida (Algoritmo 2) ---
    H_test, _ = solve_H_eval(
        W_fixed = W_learned,
        X_eval  = X_test,
        k       = k,
        steps   = steps_eval,
        alphaH  = configs[method]['alphaH'],
        method  = method,
        beta    = beta
    )
    
    rmse_test = compute_rmse(X_test, W_learned, H_test)
    print(f"  RMSE test                : {rmse_test:.6f}")
    
    results[method] = {
        'W': W_learned,
        'H_train': H_train,
        'H_val': H_val,
        'H_test': H_test,
        'loss_history': loss_hist,
        'rmse_train': rmse_train,
        'rmse_val': rmse_val,
        'rmse_test': rmse_test
    }

print(f"\n{'='*55}")
print("  RESUMEN COMPARATIVO")
print(f"{'='*55}")
print(f"{'Método':<12} {'RMSE Train':>12} {'RMSE Val':>12} {'RMSE Test':>12}")
print("-" * 50)
for method in methods:
    r = results[method]
    print(f"{method:<12} {r['rmse_train']:>12.6f} {r['rmse_val']:>12.6f} {r['rmse_test']:>12.6f}")


  Optimizador: GD
  Loss final (entrenamiento): 19092450.6426
  RMSE train               : 6.484359
  RMSE val                 : 76.602697
  RMSE test                : 88.866740

  Optimizador: MOMENTUM
  Loss final (entrenamiento): 3577883.3398
  RMSE train               : 2.807044
  RMSE val                 : 2.607469
  RMSE test                : 2.873503

  Optimizador: NESTEROV
  Loss final (entrenamiento): 816415.8449
  RMSE train               : 1.340886
  RMSE val                 : 1.261304
  RMSE test                : 1.517443

  RESUMEN COMPARATIVO
Método         RMSE Train     RMSE Val    RMSE Test
--------------------------------------------------
gd               6.484359    76.602697    88.866740
momentum         2.807044     2.607469     2.873503
nesterov         1.340886     1.261304     1.517443


---
## **Bloque 5: Guardar Matrices para el Integrante 1 (Reconstrucción de Audio)**

El Integrante 1 necesita las matrices `W` y `H` para reconstruir el audio vía iSTFT. Exportamos los resultados del mejor optimizador (o todos, según acuerden en el equipo).

In [7]:
# ============================================================
# EXPORTAR RESULTADOS para el Integrante 1 (reconstrucción de audio)
# y el Integrante 4 (gráficos y análisis de hiperparámetros)
# ============================================================

os.makedirs("../data/processed", exist_ok=True)

# Seleccionar el método de mejor desempeño en validación
best_method = min(results, key=lambda m: results[m]['rmse_val'])
print(f"Mejor método por RMSE val: {best_method.upper()}")

# Guardar matrices del mejor modelo
np.save("../data/processed/W_best.npy",      results[best_method]['W'])
np.save("../data/processed/H_train_best.npy", results[best_method]['H_train'])
np.save("../data/processed/H_val_best.npy",   results[best_method]['H_val'])
np.save("../data/processed/H_test_best.npy",  results[best_method]['H_test'])

# Guardar también todos los resultados para el Integrante 4
for method in methods:
    np.save(f"../data/processed/W_{method}.npy",      results[method]['W'])
    np.save(f"../data/processed/H_train_{method}.npy", results[method]['H_train'])
    np.save(f"../data/processed/loss_{method}.npy",    np.array(results[method]['loss_history']))

print("Matrices exportadas exitosamente a ../data/processed/")
print("\nArchivos generados:")
for f in sorted(os.listdir("../data/processed")):
    if f.endswith('.npy'):
        print(f"  {f}")

Mejor método por RMSE val: NESTEROV
Matrices exportadas exitosamente a ../data/processed/

Archivos generados:
  H_test_best.npy
  H_train_best.npy
  H_train_gd.npy
  H_train_momentum.npy
  H_train_nesterov.npy
  H_val_best.npy
  W_best.npy
  W_gd.npy
  W_momentum.npy
  W_nesterov.npy
  X_test.npy
  X_train.npy
  X_val.npy
  loss_gd.npy
  loss_momentum.npy
  loss_nesterov.npy
